# Triangle Plot: Original vs Modified Covariance

Compares parameter sampling space under:
- **Original**: Fisher covmat as-is (T=1, maxcorr=1.0)
- **Modified**: maxcorr=0.15, T=150 (the harder training case)

Chain files were generated by `run_chain_plots.sh` using `--chain 1` (no DVs computed).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from getdist import loadMCSamples, plots

CHAINS_DIR = '/home/grads/data/bela/cocoa/Cocoa/projects/roman_real/chains'

# These match the --paramfile stems + _{probe}_{temp} suffix added by the generator
ORIG_ROOT = f'{CHAINS_DIR}/chain_orig_params_cs_1'
MOD_ROOT  = f'{CHAINS_DIR}/chain_mod_params_cs_150'

print('Loading original chain...')
samples_orig = loadMCSamples(ORIG_ROOT, settings={'ignore_rows': '0.3'})
print(f'  Parameters: {samples_orig.getParamNames().list()}')
print(f'  N samples: {samples_orig.numrows}')

print('Loading modified chain...')
samples_mod = loadMCSamples(MOD_ROOT, settings={'ignore_rows': '0.3'})
print(f'  Parameters: {samples_mod.getParamNames().list()}')
print(f'  N samples: {samples_mod.numrows}')

In [ ]:
# Cosmological params only for the main plot (what Vivian wants to see)
COSMO_PARAMS = ['As_1e9', 'ns', 'H0', 'omegab', 'omegam']

g = plots.get_subplot_plotter(width_inch=10)
g.settings.axes_fontsize = 11
g.settings.legend_fontsize = 12

g.triangle_plot(
    [samples_orig, samples_mod],
    COSMO_PARAMS,
    filled=True,
    legend_labels=['Original (T=1, maxcorr=1.0)', 'Modified (T=150, maxcorr=0.15)'],
    contour_colors=['royalblue', 'tomato'],
)
plt.suptitle('Cosmological Parameter Space: Original vs Modified Covariance', 
             y=1.01, fontsize=13)
plt.savefig('triangle_cosmo_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: triangle_cosmo_comparison.png')

In [ ]:
# All 15 parameters — full triangle plot
ALL_PARAMS = [
    'As_1e9', 'ns', 'H0', 'omegab', 'omegam',
    'roman_DZ_S1', 'roman_DZ_S2', 'roman_DZ_S3', 'roman_DZ_S4',
    'roman_DZ_S5', 'roman_DZ_S6', 'roman_DZ_S7', 'roman_DZ_S8',
    'roman_A1_1', 'roman_A1_2'
]

g2 = plots.get_subplot_plotter(width_inch=18)
g2.settings.axes_fontsize = 7
g2.settings.legend_fontsize = 11

g2.triangle_plot(
    [samples_orig, samples_mod],
    ALL_PARAMS,
    filled=True,
    legend_labels=['Original (T=1, maxcorr=1.0)', 'Modified (T=150, maxcorr=0.15)'],
    contour_colors=['royalblue', 'tomato'],
)
plt.suptitle('All Parameters: Original vs Modified Covariance', y=1.01, fontsize=13)
plt.savefig('triangle_all_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: triangle_all_comparison.png')

In [ ]:
# Print 1D constraint widths for quick comparison (what to post in Slack)
print(f'{'Param':<18} {'Original 68% width':>22} {'Modified 68% width':>22} {'Ratio':>8}')
print('-' * 75)
for p in COSMO_PARAMS:
    try:
        lim_o = samples_orig.get1DDensity(p)
        lim_m = samples_mod.get1DDensity(p)
        # Use std as proxy for width
        std_o = samples_orig.std([p])[0]
        std_m = samples_mod.std([p])[0]
        ratio = std_o / std_m if std_m > 0 else float('nan')
        print(f'{p:<18} {std_o:>22.4f} {std_m:>22.4f} {ratio:>8.2f}x')
    except Exception as e:
        print(f'{p:<18}  ERROR: {e}')